In [158]:
# 1. Dataset Exploration
import pandas as pd
df = pd.read_csv("../datasets/LMS_Dirty_Dataset.csv")
df.head(10)
df.tail(10)
df.shape
df.columns
df.dtypes
df.describe()



,student_id,age,course_fee,discount,attendance_percent,assignment_score,final_score,rating
count,350.000000,350.000000,350.000000,350.000000,349.000000,350.000000,350.000000,350.000000
mean,175.500000,26.174286,10085.711429,9.600000,70.154728,67.011429,65.011429,3.960000
std,101.180532,5.485686,53109.214627,7.929489,16.697929,20.686773,20.556718,0.817852
min,1.000000,17.000000,-5000.000000,0.000000,40.000000,-5.000000,30.000000,3.000000
25%,88.250000,21.000000,5000.000000,0.000000,58.000000,51.000000,47.000000,3.000000
50%,175.500000,27.000000,7000.000000,10.000000,70.000000,69.000000,65.000000,4.000000
75%,262.750000,31.000000,10000.000000,20.000000,83.000000,85.750000,82.000000,5.000000
max,350.000000,35.000000,999999.000000,20.000000,100.000000,100.000000,100.000000,5.000000


In [43]:
# 2. Data Cleaning
# Finding the duplicated records
df.duplicated().sum()
# No duplicated records were found

np.int64(0)

In [46]:
# Finding the missing values
df.isna().sum() # 1 missing value detected in attendance_percentage column
print(df[df['attendance_percent'].isna()].index.tolist()) # the record with index 28 has null value in attendance_percent column
ap_mean = df["attendance_percent"].mean()
df["attendance_percent"] = df["attendance_percent"].fillna(ap_mean)
df.iloc[28] # handled the missing value with mean(70.154)

[]


student_id                          29
student_name                      Gita
course                      MERN Stack
city                        Biratnagar
age                                 29
gender                            Male
enrollment_date             2026-03-13
course_fee                        7000
discount                            10
payment_status                 Pending
attendance_percent           70.154728
assignment_score                    98
final_score                         80
rating                               5
mentor                       Samriddha
phone                       9822305489
email                 user28@gmail.com
completion_status              Ongoing
Name: 28, dtype: object

In [51]:
# Convert incorrect datatypes into their appropriate formats
df.dtypes # on exploring the data the enrollment_column has string datatype and can be changed to datetime
df["enrollment_date"] = pd.to_datetime(df["enrollment_date"]) # changed to datetime datatype
df.dtypes

student_id                     int64
student_name                     str
course                           str
city                             str
age                            int64
gender                           str
enrollment_date       datetime64[us]
course_fee                     int64
discount                       int64
payment_status                   str
attendance_percent           float64
assignment_score               int64
final_score                    int64
rating                         int64
mentor                           str
phone                            str
email                            str
completion_status                str
dtype: object

In [64]:
# Standardize inconsistent values in text columns
df["city"].unique() # "ktm" and "Kathmandu" inconsistency found
df["city"] = df["city"].replace("ktm","Kathmandu")
df["city"].unique()

df["course"].unique() # "Pythn AI" and "Python with AI/ML" inconsistency found
df["course"] = df["course"].replace("Pythn AI","Python with AI/ML")
df["course"].unique()

df["gender"].unique() # "No inconsistency found

df["payment_status"].unique() # "No inconsistency found

df["mentor"].unique() # "No inconsistency found

df["completion_status"].unique() # "No inconsistency found


<StringArray>
['Ongoing', 'Completed']
Length: 2, dtype: str

In [81]:
# Detect and correct invalid negative value
df[df["course_fee"] < 0]  # 1 neagtive value found
df[df["age"] < 0] 
df[df["discount"] < 0] 
df[df["attendance_percent"] < 0] 
df[df["assignment_score"] < 0] # 1 negative value found
df[df["final_score"] < 0] 
df[df["rating"] < 0] 

# Handling the negative value
df.loc[df["course_fee"] < 0, "course_fee"] = 0
df.loc[df["assignment_score"] < 0, "assignment_score"] = 0

df[df["course_fee"] < 0] # no negative value detected 
df[df["assignment_score"] < 0] # no negative value detected


,student_id,student_name,course,city,age,gender,enrollment_date,course_fee,discount,payment_status,attendance_percent,assignment_score,final_score,rating,mentor,phone,email,completion_status


In [123]:
# 3., 4. Outlier detection using IQR method and handling the outliers
# For course_fee column
q1 = df["course_fee"].quantile(0.25)
print(q1)
q3 = df["course_fee"].quantile(0.75)
print(q3)
median = df["course_fee"].quantile(0.50)
print(median)

iqr = q3 - q1
print(iqr)

lower_bound = q1 - (1.5 * iqr)
print(lower_bound)

upper_bound = q3 + (1.5 * iqr)
print(upper_bound)

outliers = df[(df["course_fee"] < lower_bound) | (df["course_fee"] > upper_bound)] # Detect the outliers
print(outliers) 

Handling outliers by removing the entire records
df = df.drop(outliers.index)
df

Handling outliers with the column median
df.loc[(df["course_fee"] < lower_bound) | (df["course_fee"] > upper_bound), "course_fee"] = int(median)

# Handling outliers using Winsorization
df["course_fee"].clip(lower_bound, upper_bound)


5000.0
10000.0
7000.0
5000.0
-2500.0
17500.0
    student_id student_name   course        city  age gender enrollment_date  \
8            9         Gita  Flutter  Biratnagar   31   Male      2026-05-06   
18          19         Sita  Flutter   Kathmandu   24   Male      2026-05-09   

    course_fee  discount payment_status  attendance_percent  assignment_score  \
8        -5000        10        Pending                70.0                88   
18      999999        10        Pending                41.0                75   

    final_score  rating mentor       phone             email completion_status  
8            98       4  Ishan  9869507556   user8@gmail.com         Completed  
18           51       4  Ishan  9803681891  user18@gmail.com           Ongoing  


student_id                          19
student_name                      Sita
course                         Flutter
city                         Kathmandu
age                                 24
gender                            Male
enrollment_date             2026-05-09
course_fee                      999999
discount                            10
payment_status                 Pending
attendance_percent                41.0
assignment_score                    75
final_score                         51
rating                               4
mentor                           Ishan
phone                       9803681891
email                 user18@gmail.com
completion_status              Ongoing
Name: 18, dtype: object

In [133]:
# For attendance_percent column
q1 = df["attendance_percent"].quantile(0.25)
print(q1)
median = df["attendance_percent"].quantile(0.50)
print(median)
q3 = df["attendance_percent"].quantile(0.75)
print(q3)

iqr = q3 - q1
print(iqr)

lower_bound = q1 - (1.5 * iqr)
print(lower_bound)

upper_bound = q3 + (1.5 * iqr)
print(upper_bound)

outliers = df[(df["attendance_percent"] < lower_bound) | (df["attendance_percent"] > upper_bound)]
print(outliers) # No outliers detected


58.0
70.0
83.0
25.0
20.5
120.5
Empty DataFrame
Columns: [student_id, student_name, course, city, age, gender, enrollment_date, course_fee, discount, payment_status, attendance_percent, assignment_score, final_score, rating, mentor, phone, email, completion_status]
Index: []


In [156]:
# For assignment_score column

# Removing negative value
df[df["assignment_score"] < 0]
df.loc[df["assignment_score"] < 0, "assignment_score"] = 0

q1 = df["assignment_score"].quantile(0.25)
print(q1)
median = df["assignment_score"].quantile(0.50)
print(median)
q3 = df["assignment_score"].quantile(0.75)
print(q3)

iqr = q3 - q1
print(iqr)

lower_bound = q1 - (1.5 * iqr)
print(lower_bound)

upper_bound = q3 + (1.5 * iqr)
print(upper_bound)

outliers = df[(df["assignment_score"] < lower_bound) | (df["assignment_score"] > upper_bound)]
print(outliers) # No ouliers found





51.0
69.0
85.75
34.75
-1.125
137.875
Empty DataFrame
Columns: [student_id, student_name, course, city, age, gender, enrollment_date, course_fee, discount, payment_status, attendance_percent, assignment_score, final_score, rating, mentor, phone, email, completion_status]
Index: []


In [157]:
# For final_score column

q1 = df["final_score"].quantile(0.25)
print(q1)
median = df["final_score"].quantile(0.50)
print(median)
q3 = df["final_score"].quantile(0.75)
print(q3)

iqr = q3 - q1
print(iqr)

lower_bound = q1 - (1.5 * iqr)
print(lower_bound)

upper_bound = q3 + (1.5 * iqr)
print(upper_bound)

outliers = df[(df["final_score"] < lower_bound) | (df["final_score"] > upper_bound)]
print(outliers) # No ouliers found



47.0
65.0
82.0
35.0
-5.5
134.5
Empty DataFrame
Columns: [student_id, student_name, course, city, age, gender, enrollment_date, course_fee, discount, payment_status, attendance_percent, assignment_score, final_score, rating, mentor, phone, email, completion_status]
Index: []


In [166]:
# Outlier Detection using Z-score
def find_zscore_ouliers(df, column_name):

    # Removing negative value
    df.loc[df[column_name] < 0, column_name] = 0

    mean = df[column_name].mean()
    std = df[column_name].std()

    df["z-score"] = (df[column_name] - mean) / std
    outliers_df = df[df["z-score"] > 3]

    return outliers_df


outliers_coursefee = find_zscore_ouliers(df, "course_fee")
print(df)
print(outliers_coursefee)

outliers_attendance_percent = find_zscore_ouliers(df, "attendance_percent")
print(df)
print(outliers_attendance_percent)

outliers_assignment_score = find_zscore_ouliers(df, "assignment_score")
print(df)
print(outliers_assignment_score)

outliers_final_score = find_zscore_ouliers(df, "final_score")
print(df)
print(outliers_final_score)


 
    

     student_id student_name             course        city  age  gender  \
0             1         Gita         MERN Stack   Kathmandu   17  Female   
1             2        Nabin       Data Science     Itahari   29  Female   
2             3         Gita         MERN Stack      Dharan   20  Female   
3             4         Gita  Digital Marketing     Itahari   20  Female   
4             5         Gita       Data Science     Itahari   19    Male   
..          ...          ...                ...         ...  ...     ...   
345         346         Hari              UI/UX      Dharan   21  Female   
346         347       Anisha  Digital Marketing  Biratnagar   31  Female   
347         348         Sita            Flutter  Biratnagar   19    Male   
348         349       Anisha              UI/UX     Itahari   27    Male   
349         350         Hari  Python with AI/ML      Dharan   19  Female   

    enrollment_date  course_fee  discount payment_status  attendance_percent  \
0      